# 01 — Postgres to Parquet

Connects to the MusicBrainz PostgreSQL database via DuckDB and exports the core tables to parquet files. This is the first step in the pipeline — all downstream notebooks depend on these files.

**What it does:**
- Opens a DuckDB connection to the local MusicBrainz Postgres instance using credentials from `.env`
- Builds a `valid_albums` scope (studio + official-live + single-artist best-of; see the scope cell) and runs filtered SQL queries to extract those albums, their tags, ratings, labels, country, and artist relationships
- Writes each table to `data/` as a ZSTD-compressed parquet file

**Inputs:** MusicBrainz PostgreSQL database (credentials in `.env`), `musicbrainz.duckdb`

**Outputs to `data/`:** `mb_artist.parquet`, `mb_artist_tag.parquet`, `mb_artist_ratings.parquet`, `mb_artist_credit.parquet`, `mb_album.parquet`, `mb_album_tag.parquet`, `mb_album_ratings.parquet`, `mb_album_country.parquet`, `mb_album_label.parquet`, `mb_album_artists.parquet`, `mb_release_year.parquet`

**Run before:** `02-parquet-to-dataframes.ipynb`

## Setup: imports, environment variables, and DuckDB connection

Loads credentials from the `.env` file and uses them to open a persistent DuckDB database (`musicbrainz.duckdb`). DuckDB is used here as a query engine rather than a primary store — it acts as a bridge between Postgres and the parquet output files.

The `INSTALL postgres` / `LOAD postgres` step downloads and activates DuckDB's Postgres scanner extension (only needed once per environment). The `ATTACH` command then makes the live MusicBrainz Postgres database available inside DuckDB as a virtual schema named `mb_pg`, which all subsequent queries reference as `mb_pg.musicbrainz.<table>`.

In [ ]:
import os
import duckdb
from dotenv import load_dotenv

load_dotenv()

pg_host = os.getenv('PG_HOST', 'localhost')
pg_port = os.getenv('PG_PORT', '5432')
pg_dbname = os.getenv('PG_DBNAME', 'musicbrainz_db')
pg_user = os.getenv('PG_USER')
pg_password = os.getenv('PG_PASSWORD')

duck_con = duckdb.connect("../musicbrainz.duckdb")

duck_con.execute("""
INSTALL postgres;
LOAD postgres;
""")

duck_con.execute(f"""
ATTACH IF NOT EXISTS 'host={pg_host} port={pg_port} dbname={pg_dbname} user={pg_user} password={pg_password}'
AS mb_pg
(TYPE postgres, READ_ONLY);
""")

## Define album scope → `valid_albums`

`release_group.type` is only the **primary** type, so `type = 1` (Album) still includes
live recordings, compilations, remixes, etc. via the *secondary* type tables, and includes
bootlegs (a `release.status`, not a release-group type). This cell builds a single
`valid_albums` table that every album export below filters against, so the scope is defined
in one place.

Scope — primary type = Album, **must have at least one Official release** (this drops
bootlegs across the board), and one of:
- **Studio albums** — no secondary types at all
- **Live albums** — secondary type Live (6) (already required Official, so no bootleg concerts)
- **Best-of** — secondary type Compilation (1) **and** credited to a single artist
  (`artist_credit <> 1`); Various-Artists compilations are dropped

Anything carrying another secondary type (Soundtrack, Remix, DJ-mix, Demo, Spokenword,
Interview, Audiobook, Mixtape, …) is excluded.

Secondary type IDs: 1 Compilation · 2 Soundtrack · 6 Live · 7 Remix · 8 DJ-mix (MusicBrainz standard).
Release status IDs: 1 Official · 3 Bootleg. Validated example: U2 drops from 1,004 release
groups to 45 — studio albums, official live albums, and official best-ofs only.

In [ ]:
# Build the canonical album scope once; every album export below joins to this table.
duck_con.execute("""
CREATE OR REPLACE TABLE valid_albums AS
WITH sec AS (
    SELECT j.release_group                          AS rg_id,
           BOOL_OR(j.secondary_type = 6)            AS is_live,
           BOOL_OR(j.secondary_type = 1)            AS is_comp,
           BOOL_OR(j.secondary_type NOT IN (1, 6))  AS has_other_sec
    FROM mb_pg.musicbrainz.release_group_secondary_type_join j
    GROUP BY j.release_group
),
official AS (
    SELECT DISTINCT release_group AS rg_id
    FROM mb_pg.musicbrainz.release
    WHERE status = 1                       -- Official
)
SELECT rg.id, rg.name, rg.artist_credit
FROM mb_pg.musicbrainz.release_group rg
LEFT JOIN sec s ON s.rg_id = rg.id
WHERE rg.type = 1                                       -- primary type = Album
  AND rg.id IN (SELECT rg_id FROM official)             -- must have an Official release (drops bootlegs)
  AND COALESCE(s.has_other_sec, FALSE) = FALSE          -- no soundtrack/remix/dj-mix/demo/etc.
  AND (
        s.rg_id IS NULL                                 -- studio album (no secondary types)
     OR s.is_live                                       -- live album (already required Official above)
     OR (s.is_comp AND rg.artist_credit <> 1)           -- single-artist best-of (drops VA compilations)
      );
""")

# Index for fast joins from the export queries below
duck_con.execute("CREATE INDEX IF NOT EXISTS idx_valid_albums_id ON valid_albums (id);")

n = duck_con.sql("SELECT COUNT(*) FROM valid_albums").fetchone()[0]
print(f"valid_albums scope: {n:,} release groups")

## Export: artists → `mb_artist.parquet`

Pulls a subset of fields from the MusicBrainz `artist` table — ID, name, the year the artist began activity (`begin_date_year`), their type (person, group, etc.), area, and gender. No filters are applied here; all artists are exported because the downstream album data can reference any of them, and we want the lookup table to be complete.

Output columns: `id`, `name`, `artist_year`, `type`, `area`, `gender`.

In [ ]:
#import artists
duck_con.execute(f"""
COPY (
    SELECT
        id,
        name,
        begin_date_year AS artist_year,
        type,
        area,
        gender
    FROM mb_pg.musicbrainz.artist
    ORDER BY id
)
TO '../data/mb_artist.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

## Export: artist tags → `mb_artist_tag.parquet`

Exports the many-to-many mapping between artists and tags from `artist_tag`. Each row represents a user-applied genre/style tag on an artist, along with a `tag_count` that reflects how many MusicBrainz users applied that tag. No count filter is applied — even tags with low counts are included so downstream processing can decide its own threshold.

Output columns: `artist_id`, `tag_id`, `tag_count`.

In [ ]:
#import artist tags
duck_con.execute(f"""
COPY (
    SELECT
        artist AS artist_id,
        tag AS tag_id,
        count AS tag_count
    FROM mb_pg.musicbrainz.artist_tag
    ORDER BY artist
)
TO '../data/mb_artist_tag.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

## Export: artist ratings → `mb_artist_ratings.parquet`

Reads from `artist_meta`, which stores aggregate community rating data for each artist. Only artists with a non-null `rating` are included — unrated artists are not useful for recommendation scoring. Results are ordered by `rating_count` descending so the most-rated artists appear first, which is useful for quick inspection.

Output columns: `artist_id`, `rating`, `rating_count`.

In [ ]:
#import artist ratings
duck_con.execute(f"""
COPY (
    SELECT
        id AS artist_id,
        rating,
        rating_count
    FROM mb_pg.musicbrainz.artist_meta
    WHERE rating IS NOT NULL
    ORDER BY rating_count DESC
)
TO '../data/mb_artist_ratings.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

## Export: albums → `mb_album.parquet`

Exports the album table from the `valid_albums` scope built above — studio albums, official
live albums, and single-artist best-of collections (see the scope cell for the exact rules).
This is the primary scope boundary for the entire project; every album table below joins to
`valid_albums`.

The `artist_credit` column is a foreign key into the `artist_credit` table, which maps to one
or more actual artists. It is not a direct artist ID.

Output columns: `id`, `name`, `artist_credit`.

In [ ]:
#import albums
duck_con.execute(f"""
COPY (
    SELECT id, name, artist_credit
    FROM valid_albums
    ORDER BY id
)
TO '../data/mb_album.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

## Export: album tags → `mb_album_tag.parquet`

Exports genre/style tags for albums from `release_group_tag`, joining back to `release_group` to apply the `type = 1` album filter. An additional filter `t.count > 0` removes any tag rows that have been down-voted to zero by the community, keeping only tags with at least one net positive vote.

Note: `tag_id` here is a numeric foreign key into the MusicBrainz `tag` table (not exported separately). The human-readable tag name is resolved in a later notebook.

Output columns: `album_id`, `tag_id`, `tag_count`.

In [ ]:
#import album tags
duck_con.execute(f"""
COPY (
    SELECT
        t.release_group AS album_id,
        t.tag AS tag_id,
        t.count AS tag_count
    FROM mb_pg.musicbrainz.release_group_tag t
    JOIN valid_albums va ON t.release_group = va.id
    WHERE t.count > 0
    ORDER BY t.release_group
)
TO '../data/mb_album_tag.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

## Export: album ratings → `mb_album_ratings.parquet`

Reads community ratings from `release_group_meta`, joining to `release_group` to filter for albums only (`type = 1`) and to exclude unrated entries (`rating IS NOT NULL`). Ordered by `rating_count` descending so the most-reviewed albums appear first.

The `rating` value is an integer on a 0–100 scale (MusicBrainz stores ratings as 1–5 stars multiplied by 20). `rating_count` is the number of users who submitted a rating.

Output columns: `album_id`, `rating`, `rating_count`.

In [ ]:
#import album ratings
duck_con.execute(f"""
COPY (
    SELECT
        m.id AS album_id,
        m.rating,
        m.rating_count
    FROM mb_pg.musicbrainz.release_group_meta m
    JOIN valid_albums va ON m.id = va.id
    WHERE m.rating IS NOT NULL
    ORDER BY m.rating_count DESC
)
TO '../data/mb_album_ratings.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

## Export: album country and release year → `mb_album_country.parquet`

This is the most complex export. In MusicBrainz, a `release_group` (album) can have many individual `release` records — one per country, format, or edition. This query picks exactly one release per album using `DISTINCT ON (r.release_group)`, choosing the earliest known physical release by sorting on `date_year ASC NULLS LAST`, then `date_month`, then `date_day`. NULLs are pushed last so that partially-dated releases are preferred over completely undated ones.

The result gives one row per album with the country and year of its first known release, plus the language of that release. This is used downstream as the canonical release year for an album.

Output columns: `album_id`, `language`, `country`, `album_year`.

In [ ]:
#import album country
duck_con.execute(f"""
COPY (
    SELECT DISTINCT ON (r.release_group)
        r.release_group AS album_id,
        r.language,
        rc.country,
        rc.date_year AS album_year
    FROM mb_pg.musicbrainz.release r
    JOIN mb_pg.musicbrainz.release_country rc ON r.id = rc.release
    JOIN valid_albums va ON r.release_group = va.id
    ORDER BY r.release_group,
             rc.date_year ASC NULLS LAST,
             rc.date_month ASC NULLS LAST,
             rc.date_day ASC NULLS LAST
)
TO '../data/mb_album_country.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

## Export: album label → `mb_album_label.parquet`

Associates each album with a record label and that label's most prominent tag (genre/descriptor). The query joins four tables: `release_label` → `release` → `release_group` → `label` → `label_tag`. The `type = 1` filter on `release_group` keeps only albums.

Like the country export, `DISTINCT ON (r.release_group)` picks one row per album. The tie-breaking `ORDER BY lt.count DESC NULLS LAST` selects the label tag with the highest community vote count, so the most-agreed-upon descriptor for that label is kept.

This is useful downstream for inferring genre signals from the label when an album has few direct tags of its own.

Output columns: `album_id`, `label_id`, `label_type`, `tag_id`, `tag_count`.

In [ ]:
#import album label
duck_con.execute(f"""
COPY (
    SELECT DISTINCT ON (r.release_group)
        r.release_group AS album_id,
        rl.label AS label_id,
        l.type AS label_type,
        lt.tag AS tag_id,
        lt.count AS tag_count
    FROM mb_pg.musicbrainz.release_label rl
    JOIN mb_pg.musicbrainz.release r ON rl.release = r.id
    JOIN valid_albums va ON r.release_group = va.id
    JOIN mb_pg.musicbrainz.label l ON rl.label = l.id
    JOIN mb_pg.musicbrainz.label_tag lt ON rl.label = lt.label
    ORDER BY r.release_group,
             lt.count DESC NULLS LAST
)
TO '../data/mb_album_label.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

## Export: artist credits → `mb_artist_credit.parquet`

In MusicBrainz, `artist_credit` is an intermediate entity that sits between a release and one or more artists — it captures credited names and join phrases (e.g. "feat.", "&"). This query exports the credit records for release groups that are NOT standard albums (`rg.type != 1`), i.e. singles, EPs, and other non-album releases.

Each row in the output represents one artist's position within a credit group: their credited name, position index, join phrase to the next artist, and the underlying `artist_id`.

Note the inverted filter here (`type != 1`) compared to every other export. This is intentional — album-to-artist mapping is handled separately in the next cell with more sophisticated deduplication logic.

Output columns: `artist_credit`, `name`, `artist_count`, `ref_count`, `position`, `artist_id`, `artist_name`, `join_phrase`.

In [ ]:
#import artist credit
duck_con.execute(f"""
COPY (
    SELECT
        ac.id AS artist_credit,
        ac.name,
        ac.artist_count,
        ac.ref_count,
        acn.position,
        acn.artist AS artist_id,
        acn.name AS artist_name,
        acn.join_phrase
    FROM mb_pg.musicbrainz.artist_credit ac
    JOIN mb_pg.musicbrainz.artist_credit_name acn ON ac.id = acn.artist_credit
    JOIN mb_pg.musicbrainz.release_group rg ON rg.artist_credit = ac.id
    WHERE rg.type != 1
    ORDER BY ac.id
)
TO '../data/mb_artist_credit.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

## Export: album-to-artist mapping → `mb_album_artists.parquet`

Builds a clean one-row-per-album mapping from album to primary artist, handling the "Various Artists" edge case. In MusicBrainz, compilations credited to "Various Artists" use `artist_id = 1` or `artist_credit = 1` as a sentinel value.

The CTE `artist_ranked` uses `ROW_NUMBER()` to rank artists per album: real named artists (not VA) are ranked first (`CASE WHEN acn.artist != 1 AND acn.artist_credit != 1 THEN 0 ELSE 1 END ASC`), then by their credited position. The outer query then takes only `rn = 1` (the top-ranked artist per album), and uses a `CASE` expression to null out `artist_id` and `artist_name` for any VA rows that still make it through.

The result is one row per album with its primary non-VA artist, or NULL artist fields if the album is genuinely a Various Artists release. This is the main join table used downstream to link albums to artist features.

Output columns: `album_id`, `album_name`, `artist_id`, `artist_name`.

## Export: release group year → `mb_release_year.parquet`

Pulls `first_release_date_year` from `release_group_meta` for every album in the `valid_albums`
scope. This is MusicBrainz's canonical first-release year for a release group — the earliest
known date across all individual releases (any country, any format). It is the most semantically
correct source for 'when was this album first made'.

Only rows where `first_release_date_year IS NOT NULL` are included; release groups with no
known year are excluded from this file and handled downstream as NULL.

Output columns: `album_id`, `release_group_meta_year`.

In [ ]:
#import release group year
duck_con.execute(f"""
COPY (
    SELECT
        va.id AS album_id,
        rgm.first_release_date_year AS release_group_meta_year
    FROM valid_albums va
    JOIN mb_pg.musicbrainz.release_group_meta rgm ON rgm.id = va.id
    WHERE rgm.first_release_date_year IS NOT NULL
    ORDER BY va.id
)
TO '../data/mb_release_year.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")

n = duck_con.sql("SELECT COUNT(*) FROM read_parquet('../data/mb_release_year.parquet')").fetchone()[0]
print(f"mb_release_year: {n:,} rows")

In [ ]:
#import album artists
duck_con.execute(f"""
COPY (
    WITH artist_ranked AS (
        SELECT
            rg.id AS album_id,
            rg.name AS album_name,
            acn.artist AS artist_id,
            acn.name AS artist_name,
            CASE WHEN acn.artist = 1 OR acn.artist_credit = 1 THEN 1 ELSE 0 END AS is_va,
            ROW_NUMBER() OVER (
                PARTITION BY rg.id
                ORDER BY
                    CASE WHEN acn.artist != 1 AND acn.artist_credit != 1 THEN 0 ELSE 1 END ASC,
                    acn.position ASC
            ) AS rn
        FROM valid_albums rg
        JOIN mb_pg.musicbrainz.artist_credit_name acn ON rg.artist_credit = acn.artist_credit
    )
    SELECT DISTINCT
        album_id,
        album_name,
        CASE WHEN is_va = 0 THEN artist_id END AS artist_id,
        CASE WHEN is_va = 0 THEN artist_name END AS artist_name
    FROM artist_ranked
    WHERE rn = 1
    ORDER BY album_id
)
TO '../data/mb_album_artists.parquet'
(FORMAT 'PARQUET', COMPRESSION 'ZSTD');
""")